# Phase 0: Foundation

One tool (`read_file`), one loop. See [README.md](README.md) for the full explanation.

**The one idea:** the LLM only produces text. Our Python code does all the actual work.

## 1. Setup

Connect to the model. `LLM_API_KEY`, `LLM_BASE_URL` and `LLM_MODEL` come from the `.env` file at the repo root.
Nothing is sent yet; `client` just remembers *where* the model lives.

In [1]:
import json
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"])
MODEL = os.environ["LLM_MODEL"]

## 2. A safe playground

A small file for the agent to look at, in a `sandbox/` folder so it never touches anything real.

In [2]:
sandbox = Path("sandbox")
sandbox.mkdir(exist_ok=True)
(sandbox / "hello.py").write_text('name = "world"\nprint(f"hello {name}")\n');

## 3. The tool

A plain Python function, and the agent's only way to see anything.
It returns a dict (easy to turn into text for the model) and returns errors instead of crashing, so the model can react to them.

In [3]:
def read_file(filename: str) -> dict:
    """Read a text file and return its contents."""
    path = Path(filename)
    if not path.exists():
        return {"error": f"File not found: {filename}"}
    return {"filename": filename, "content": path.read_text()}

In [4]:
print(read_file("sandbox/hello.py"))
print(read_file("sandbox/nope.txt"))

{'filename': 'sandbox/hello.py', 'content': 'name = "world"\nprint(f"hello {name}")\n'}
{'error': 'File not found: sandbox/nope.txt'}


## 4. The system prompt

The **only** way the model learns the tool exists. We're inventing a tiny text protocol:
*"if you want a file, write this exact line, and file contents will show up."*

In [5]:
SYSTEM_PROMPT = """You are a helpful coding assistant.

You have ONE tool:
  read_file(filename) - returns the contents of a text file.

To use it, reply with exactly one line and nothing else:
tool: read_file({"filename": "path/to/file"})

After you receive a tool_result(...) message, use it to continue the task.
If you don't need the tool, just answer normally."""

## 5. Calling the model

Send the **whole conversation** and get the reply back as text. The model has no memory, so every call includes everything said so far.

In [6]:
def call_llm(conversation: list) -> str:
    response = client.chat.completions.create(model=MODEL, messages=conversation)
    return response.choices[0].message.content

In [7]:
call_llm([{"role": "user", "content": "What's 2+2? Answer in one word."}])

'Four'

## 6. Spotting tool requests

Turn the model's free text into structured data: a dict of arguments if the reply contains our `tool: read_file({...})` line, otherwise `None`.

In [8]:
def parse_tool_call(reply: str):
    prefix = "tool: read_file("
    for line in reply.splitlines():
        line = line.strip()
        if line.startswith(prefix) and line.endswith(")"):
            try:
                return json.loads(line[len(prefix):-1])
            except json.JSONDecodeError:
                return None
    return None

In [9]:
print(parse_tool_call('tool: read_file({"filename": "sandbox/hello.py"})'))
print(parse_tool_call("The answer is 4."))

{'filename': 'sandbox/hello.py'}
None


## 7. The agent loop

Each pass: call the model → did it ask for a tool? → if not, that's the answer; if so, run the tool, add the result to the conversation, and go again.

`max_steps` is a safety limit (like `max_iter` in scikit-learn) so a confused model can't loop forever.

In [10]:
conversation = [{"role": "system", "content": SYSTEM_PROMPT}]

def ask(user_message: str, max_steps: int = 5) -> str:
    conversation.append({"role": "user", "content": user_message})

    for step in range(max_steps):
        reply = call_llm(conversation)                          # 1. ask the model
        conversation.append({"role": "assistant", "content": reply})

        args = parse_tool_call(reply)                           # 2. did it ask for a tool?
        if args is None:
            print("Assistant:", reply)                          # 3a. no -> final answer
            return reply

        print(f"  [step {step + 1}] model asked for read_file({args})")
        result = read_file(args.get("filename", ""))            # 3b. yes -> run the real function
        conversation.append(                                    # 4. send the result back
            {"role": "user", "content": f"tool_result({json.dumps(result)})"}
        )

    print("Stopped: too many tool calls.")

## 8. Try it

In [11]:
ask("What will sandbox/hello.py print when I run it?");

  [step 1] model asked for read_file({'filename': 'sandbox/hello.py'})


Assistant: When you run sandbox/hello.py, it will print:

```
hello world
```

The script defines a variable `name` with the value "world", then uses an f-string to print "hello" followed by the value of `name`.


In [12]:
ask("If I changed the name to Alex, what would it print?");

Assistant: If you changed the name to "Alex" in the script, it would print:

```
hello Alex
```

The output depends on the value of the `name` variable, so changing it from "world" to "Alex" would change the printed output accordingly.


## 9. Look inside

The `conversation` list *is* the agent's memory. Match each message to the walkthrough table in the README.

In [13]:
for i, m in enumerate(conversation):
    print(f"{i}  {m['role']:9}  {m['content'][:80]!r}")

0  system     'You are a helpful coding assistant.\n\nYou have ONE tool:\n  read_file(filename) - '
1  user       'What will sandbox/hello.py print when I run it?'
2  assistant  "I'll check the contents of the sandbox/hello.py file to see what it will print w"
3  user       'tool_result({"filename": "sandbox/hello.py", "content": "name = \\"world\\"\\nprint'
4  assistant  'When you run sandbox/hello.py, it will print:\n\n```\nhello world\n```\n\nThe script d'
5  user       'If I changed the name to Alex, what would it print?'
6  assistant  'If you changed the name to "Alex" in the script, it would print:\n\n```\nhello Alex'


## Your turn

Predict what happens, then run:

1. `ask("What's the capital of France?")`: does it use the tool?
2. `ask("What does sandbox/missing.py contain?")`: what does the model see?
3. `conversation = conversation[:1]`, then a follow-up question. Watch it forget.